# Exploratory Data Analysis

This notebook wil explore the historical regular-season games from the seasons 2018-19 through 2025-26.

The objective for this analysis is to identify any relationships that may be important for predicting game outcomes using only pre-game information. 



In [2]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd


In [3]:
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent

games = pd.read_parquet(root/'data'/'processed'/'games_all.parquet')

games['game_date'] = pd.to_datetime(games['game_date'])
print(games.shape)
display(games.head())

(9519, 10)


,game_id,game_date,season,home_id,away_id,home_team,away_team,home_score,away_score,home_win
0,401070213,2018-10-16,2019,2,20,BOS,PHI,105,87,1
1,401070214,2018-10-16,2019,9,25,GS,OKC,108,100,1
2,401070215,2018-10-17,2019,10,3,HOU,NO,112,131,0
3,401070216,2018-10-17,2019,21,6,PHX,DAL,121,100,1
4,401070693,2018-10-17,2019,30,15,CHA,MIL,112,113,0


## Analysis 1: Home-court advantage


How frequently do home teams win? How does this change between seasons?

In [4]:
# Calculate the home-team win rate for each season.

home_advantage = games.groupby('season', as_index=False).agg(
    total_games = ('game_id', 'count'),
    home_win_rate = ('home_win', 'mean')
)

display(home_advantage)

,season,total_games,home_win_rate
0,2019,1230,0.592683
1,2020,1059,0.551464
2,2021,1080,0.543519
3,2022,1230,0.543902
4,2023,1230,0.580488
5,2024,1230,0.543089
6,2025,1230,0.543902
7,2026,1230,0.554472


In [7]:
home_advantage_plot = alt.Chart(home_advantage).mark_line(point=True).encode(
    x=alt.X('season:O', title='Season ending year', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('home_win_rate:Q',
             title='Home win rate',
             axis=alt.Axis(format='.0%'),
             scale=alt.Scale(domain=[0,1])),
    tooltip=[
        'season:O',
        'total_games:Q',
        alt.Tooltip('home_win_rate:Q', format='.1%')
    ]
).properties(
    title='NBA Home Win Rate by Season',
    width=650,
    height=350
)

home_advantage_plot

alt.Chart(...)

Throughout these eight seasons, the home-win rate is consistently over 50%, albeit not by a large margin. Even during the majorly affected COVID seasons that lasted throughout the 2019-20 and 2020-21 season, the home-win rate remained consistent with other years, sitting right in between 50-60%. 

Things to note about the COVID-affected seasons:
- During the 2019-20 season, the season was suspended in March 2020, with playing resumed in July 2020. However, playing resume in July 2020 at the Disney World bubble. Fan attendance was 0 at this time. 
- During the 2020-21 season, the season was delayed and shortened to a 72 game schedule with empty/limited capacity arenas. Toronto Raptors temporarily played home games in Tampa, Florida. 
- During the 2021-22 season, it returned to a full 82-game schedule. However, it was impacted by the Omicron variant and vaccine mandates which restricted unvaccinated players.

Despite turbulent variables, the home-win rate stayed relatively consistent historically to be a stable statistic to take incorporate into our prediction model. 

## Analysis 2: Home-court advantage by team

Do certain teams benefit more from playing on their home court?

In [ ]:
home_stats = games.groupby('home_team').agg(
    home_games=('game_id','count'),
    home_wins=('home_win','sum'),
    home_win_rate= ('home_win', 'mean')
).reset_index().rename(columns={'home_team':'team'})


away_stats = games.assign(away_win=1-games['home_win']).groupby('away_team').agg(
    away_games=('game_id','count'),
    away_wins=('away_win', 'sum'),
    away_win_rate=('away_win','mean')
).reset_index().rename(columns={'away_team':'team'})

team_advantage = home_stats.merge(away_stats, on='team', validate='one_to_one')

team_advantage['win_rate_difference'] = team_advantage['home_win_rate'] - team_advantage['away_win_rate']

In [12]:
team_advantage_plot = alt.Chart(team_advantage).mark_bar().encode(
    y=alt.Y('team:N', sort='-x', title='Team'),
    x=alt.X('win_rate_difference:Q', title='Home minus Away win rate', axis=alt.Axis(format='.0%')),
    tooltip=[
        'team:N',
        alt.Tooltip('home_win_rate:Q', format='.1%'),
        alt.Tooltip('away_win_rate:Q', format='.1%')
    ]
).properties(title='Home vs. Away Win Rate by Team', width=750, height=350)

team_advantage_plot

alt.Chart(...)

This chart represents the difference in the team's home win rate against the team's away win rate. Clearly, all teams seem to have a higher home win rate than away_win rate with the lowest difference just over 4% and the greatest difference just over 18%. 